# Load sc distances

In [ ]:
sc_dis_file = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/cluster.LINK/astro/compare_to_1cell/matrix2d/astro.distances.per_locus.csv.new'
struct_infer_file = '/net/gs/vol1/home/gesine/noble_lab/projects/2015_gesine_diploid/results/liu2025/test_astro_1e6_750nm/infer_ua_filter0perc.singleres/struct_inferred.000.coords'

sc_dis = load_sc_distances(sc_dis_file, scale=True, verbose=True).drop('same_molecule', axis=1)

Loading sc distances...


# Code

In [ ]:
import os
import re
import ast
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from topsy.analysis.utils import get_nghbr_dis_var, _get_mse


def make_diploid_lengths_df(lengths_df):
    if isinstance(lengths_df, str):
        lengths_df = pd.read_csv(lengths_df, sep="\t")
        lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]

    tmp = lengths_df.copy()
    tmp['idx'] = tmp.idx_genome
    tmp['hmlg'] = 1
    diploid_lengths_df = tmp.copy()
    diploid_lengths_df['idx'] += len(lengths_df)
    diploid_lengths_df['hmlg'] += 1
    diploid_lengths_df = pd.concat([
        tmp, diploid_lengths_df]).reset_index(drop=True)
    diploid_lengths_df['mol'] = diploid_lengths_df.chrom + '.' + \
        diploid_lengths_df.hmlg.astype(str)

    return diploid_lengths_df


def make_matrix_df(lengths_df, ploidy=2, matrix_dict=None):
    if isinstance(lengths_df, str):
        lengths_df = pd.read_csv(lengths_df, sep="\t")
        lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]

    lengths = lengths_df.groupby('chrom').size().sort_values(
        ascending=False).values
    nbeads = lengths.sum() * ploidy

    if ploidy == 2:
        lengths_df = make_diploid_lengths_df(lengths_df)

    rows, cols = np.triu_indices(nbeads, 1)
    df = pd.DataFrame()
    df['i.idx'] = lengths_df.idx.values[rows]
    df['j.idx'] = lengths_df.idx.values[cols]
    df['i.idx_chrom'] = lengths_df.idx_chrom.values[rows]
    df['j.idx_chrom'] = lengths_df.idx_chrom.values[cols]
    df['i.chrom'] = lengths_df.chrom.values[rows]
    df['j.chrom'] = lengths_df.chrom.values[cols]
    if ploidy == 2:
        df['i.idx_ambig'] = lengths_df.idx_genome.values[rows]
        df['j.idx_ambig'] = lengths_df.idx_genome.values[cols]
        df['i.hmlg'] = lengths_df.hmlg.values[rows]
        df['j.hmlg'] = lengths_df.hmlg.values[cols]

    # diffM, sameC-sameH, sameC-diffH, diffC-sameH, diffC-diffH
    sameC = df['i.chrom'] == df['j.chrom']
    if ploidy == 1:
        df['mask.sameM'] = sameC
        df['mask.diffM'] = ~sameC
    else:
        sameH = df['i.hmlg'] == df['j.hmlg']
        df['mask.diffM'] = (~sameC) | (~sameH)
        df['mask.sameC-sameH'] = ~df['mask.diffM']  # aka sameM
        df['mask.sameC-diffH'] = sameC & (~sameH)  # included in diffM
        df['mask.diffC-sameH'] = (~sameC) & sameH  # included in diffM
        df['mask.diffC-diffH'] = (~sameC) & (~sameH)  # included in diffM

    df.index = list(map(tuple, np.stack(
        [df['i.idx'], df['j.idx']], axis=1).tolist()))

    if matrix_dict is not None:
        for name, matrix in matrix_dict.items():
            # df[name] = matrix[triu]
            df[name] = matrix[(df['i.idx'].values, df['j.idx'].values)]

    return df


def establish_hmlg_order(matrix_df, res_df, swap_df):
    mask = matrix_df.dis_true.isnull() & matrix_df['mask.sameC-sameH']

    n = matrix_df.loc[
        matrix_df['i.hmlg'] == 2, 'i.idx'].values[0] - matrix_df.loc[
        matrix_df['i.hmlg'] == 2, 'i.idx_ambig'].values[0]

    # Error score that determines whether homologs should be swapped
    if len(res_df.columns) == 1:
        err_score = res_df.columns.values[0]
    else:
        err_score = 'vs_all'

    swap_chrom = res_df[mask].groupby(matrix_df.loc[mask, 'i.chrom']).apply(
        np.mean) > swap_df[mask].groupby(
        matrix_df.loc[mask, 'i.chrom']).apply(np.mean)
    chrom_to_swap = swap_chrom[swap_chrom[err_score]].index

    for chrom in chrom_to_swap:
        for locus in ('i', 'j'):
            chrom_mask = mask & (matrix_df[f"{locus}.chrom"] == chrom)
            matrix_df.loc[chrom_mask, 'swapped'] += 1
            # Swap the homolog labels
            matrix_df.loc[chrom_mask, f"{locus}.hmlg"] = np.invert((
                matrix_df.loc[chrom_mask, f"{locus}.hmlg"] - 1).astype(
                bool)).astype(int) + 1
            # Update locus idx labels to match
            matrix_df.loc[chrom_mask, f"{locus}.idx"] = matrix_df.loc[
                chrom_mask, f"{locus}.idx_ambig"] + n * (
                    matrix_df.loc[chrom_mask, f"{locus}.hmlg"] - 1)
        # Incorporate similarity scores obtained from chrom-swapped data
        res_df[mask & (matrix_df['i.chrom'] == chrom)] = swap_df[mask & (
            matrix_df['i.chrom'] == chrom)]
    if len(chrom_to_swap):
        # Update DataFrame index to match
        matrix_df.index = list(map(tuple, np.stack(
            [matrix_df['i.idx'], matrix_df['j.idx']], axis=1).tolist()))

    for col in res_df.columns:
        matrix_df[col] = res_df[col]

    return matrix_df


def distances_infer_vs_true(matrix_df, sc_dis, guess_hmlg_order=True,
                            verbose=True):

    for col in ['vs_all', 'vs_mean', 'vs_med', 'dis_true_mean', 'dis_true_med']:
        if col not in matrix_df.columns:
            matrix_df[col] = np.nan
    if 'swapped' not in matrix_df.columns:
        matrix_df['swapped'] = 0

    if verbose:
        if guess_hmlg_order:
            print("Comparing intra-molecular distance bins and matching up"
                  " homologs", flush=True)
        else:
            print("Comparing all distance bins", flush=True)

    if guess_hmlg_order:
        res_df = matrix_df[['vs_all', 'vs_mean', 'vs_med']].copy()
        swap_df = res_df.copy()
    else:
        res_df = matrix_df
        swap_df = None

    matrix_df = matrix_df.loc[sc_dis.index]  # Ordered to match sc_dis

    mask = matrix_df.dis_true.isnull()
    if guess_hmlg_order:  # Only using intra-molecular to guess homolog order
        mask = mask & matrix_df['mask.sameC-sameH']

    matrix_df.loc[mask, 'dis_true_mean'] = sc_dis.loc[mask, 'dis_mean']
    matrix_df.loc[mask, 'dis_true_med'] = sc_dis.loc[mask, 'dis_med']
    dis_true = sc_dis.loc[mask, 'dis']

    dis_infer = matrix_df.loc[mask, 'dis_infer']
    res_df.loc[mask, 'vs_mean'] = np.square(
        dis_infer - matrix_df.loc[mask, 'dis_true_mean'])
    res_df.loc[mask, 'vs_med'] = np.square(
        dis_infer - matrix_df.loc[mask, 'dis_true_med'])
    res_df.loc[mask, 'vs_all'] = np.square(dis_infer - dis_true).apply(np.mean)
    if guess_hmlg_order:
        dis_infer = matrix_df.loc[mask, 'dis_infer.swap']  # Homologs swapped
        swap_df.loc[mask, 'vs_mean'] = np.square(
            dis_infer - matrix_df.loc[mask, 'dis_true_mean'])
        swap_df.loc[mask, 'vs_med'] = np.square(
            dis_infer - matrix_df.loc[mask, 'dis_true_med'])
        swap_df.loc[mask, 'vs_all'] = np.square(dis_infer - dis_true).apply(
            np.mean)

    if guess_hmlg_order:
        matrix_df = establish_hmlg_order(
            matrix_df, res_df=res_df, swap_df=swap_df)
        return distances_infer_vs_true(
            matrix_df, sc_dis=sc_dis, guess_hmlg_order=False, verbose=verbose)

    return matrix_df


def get_other_struct_features(matrix_df, dis_df, ploidy=2):
    matrix_df = matrix_df[dis_df.index]  # Sort/filter to match dis_df

    if matrix_df['mask.diffM'].sum():  # Only using intra-molecular distances
        sameM = ~matrix_df['mask.diffM'].values
        matrix_df = matrix_df[sameM]
        dis_df = dis_df[sameM]

    results = {}

    # Characterize distances between neighboring beads
    mask_nghbr = (matrix_df['i.chrom'] - matrix_df['j.chrom']).abs() == 1
    results['nghbr_dis_mean'] = dis_df[mask_nghbr].mean(axis=0)
    results['nghbr_dis_var'] = dis_df[mask_nghbr].apply(
        get_nghbr_dis_var, axis=0)

    # How different are the homologs? MSE of Distance error between homologs
    if ploidy == 2:
        n = matrix_df.loc[
            matrix_df['i.hmlg'] == 2, 'i.idx'].values[0] - matrix_df.loc[
            matrix_df['i.hmlg'] == 2, 'i.idx_ambig'].values[0]
        idx_hmlg1 = matrix_df[matrix_df['i.hmlg'] == 1].index
        idx_hmlg2 = list(map(tuple, (
            np.array(idx_hmlg1.values.tolist()) + n).tolist()))
        results['disterr_btwn_hmlg'] = np.sqrt(np.nanmean(np.square(
            dis_df[idx_hmlg1].values - dis_df[idx_hmlg2].values), axis=0))

    return results


def compare_infer_vs_true(struct_infer_file, sc_dis, hmlg_order_known=None,
                          redo=False, verbose=True):

    outdir = os.path.dirname(struct_infer_file)
    struct_desc = re.sub(
        r'\.coords(\.gz)*$', '', os.path.basename(struct_infer_file)).replace(
        'struct_inferred.', 'infer').replace('struct_init.', 'init')
    outfile_perbin = os.path.join(
        outdir, f"disterror_per_bin.true_vs_{struct_desc}")
    outfile_scores = os.path.join(outdir, f"error.true_vs_{struct_desc}")
    print(outfile_perbin, flush=True)

    if (not redo) and os.path.isfile(outfile_perbin) and os.path.isfile(
            outfile_scores):
        matrix_df = pd.read_csv(outfile_perbin)
        raise NotImplementedError()

    # Load data
    matrix_df, dataset_dir = load_inferred_dis(
        struct_infer_file, hmlg_order_known=hmlg_order_known, verbose=verbose)
    if sc_dis is None:
        sc_dis = os.path.join(dataset_dir, "dis_true.per_locus.csv")
    if isinstance(sc_dis, str):
        sc_dis = load_sc_distances(sc_dis, scale=True, verbose=verbose).drop(
            'same_molecule', axis=1)
    # elif not sc_dis_are_scaled:
    #     sc_dis = scale_sc_distances(
    #         sc_dis, dataset_dir=dataset_dir, copy=True, verbose=verbose)

    if (not redo) and os.path.isfile(outfile_perbin):
        matrix_df = pd.read_csv(outfile_perbin)
    else:
        matrix_df = distances_infer_vs_true(
            matrix_df=matrix_df, sc_dis=sc_dis,
            guess_hmlg_order=not hmlg_order_known, verbose=verbose)
        matrix_df.to_csv(outfile_perbin)

    disterror_res = get_disterror_from_sq_dis(matrix_df, verbose=verbose)

    if (not redo) and os.path.isfile(outfile_scores):
        error = pd.read_csv(outfile_scores)
    else:
        raise NotImplementedError()
        # error.to_csv(outfile_scores, sep='\t', header=False)

    return matrix_df


def get_disterror_from_sq_dis(matrix_df, verbose=True):
    disterror_types = [c.replace(
        'mask.', '') for c in matrix_df.columns if c.startswith('mask.')]

    disterror_res = pd.DataFrame(
        columns=['vs_all', 'vs_mean', 'vs_med'],
        index=['all'] + disterror_types)
    for col in disterror_res.columns:
        disterror_res.loc['all', col] = matrix_df[col].mean()
        for err_type in disterror_types:
            disterror_res.loc[err_type, col] = matrix_df.loc[
                matrix_df[f"mask.{err_type}"], col].mean().sqrt()
    if verbose:
        print(disterror_res.to_string(), flush=True)
    return disterror_res


def get_dataset_dir(struct_infer_files):
    if isinstance(struct_infer_files, str):
        struct_infer_files = [struct_infer_files]

    dataset_dir = os.path.commonpath([
        os.path.dirname(x) for x in struct_infer_files])
    i = 0
    while i < 10 and not os.path.isfile(os.path.join(
            dataset_dir, 'counts.bed')):
        dataset_dir = os.path.dirname(dataset_dir)
        i += 1
    if not os.path.isfile(os.path.join(dataset_dir, 'counts.bed')):
        raise ValueError("Couldn't find directory with dataset files...")


def load_inferred_dis(struct_infer_file, hmlg_order_known=None, verbose=True):
    dataset_dir = get_dataset_dir(struct_infer_file)

    lengths_df = pd.read_csv(os.path.join(dataset_dir, 'counts.bed'), sep="\t")
    lengths_df.columns = [c.replace('#', '') for c in lengths_df.columns]
    lengths = lengths_df.groupby('chrom').size().sort_values(
        ascending=False).values
    n = lengths.sum()

    if hmlg_order_known is None:
        config_file = os.path.join(
            os.path.dirname(struct_infer_file), "config.pastis")
        counts_files = [os.path.basename(x).lower() for x in pd.read_csv(
            config_file, sep='\t', header=None, index_col=0).squeeze(
            "columns")["counts"].split(' ')]
        hmlg_order_known = any([
            ('counts_ua' in x or 'ua_counts' in x) for x in counts_files])
        if verbose:
            print("Assuming homolog ordering of inferred structure has "
                  f"{'' if hmlg_order_known else 'NOT '}been pre-established",
                  flush=True)

    struct_infer = np.loadtxt(struct_infer_file)

    matrix_dict = {'dis_infer': squareform(pdist(struct_infer))}
    if not hmlg_order_known:
        struct_infer_swap = np.append(struct_infer[n:], struct_infer[:n])
        matrix_dict['dis_infer.swap'] = squareform(pdist(struct_infer_swap))

    matrix_df = make_matrix_df(lengths_df, matrix_dict=matrix_dict)
    matrix_df = matrix_df[~matrix_df.dis_infer.isnull()]

    return matrix_df, dataset_dir


def load_sc_distances(sc_dis_file, scale=True, verbose=True):
    if os.path.isfile(f'{sc_dis_file}.gz') and not os.path.isfile(sc_dis_file):
        sc_dis_file = f'{sc_dis_file}.gz'
    if verbose:
        print("Loading sc distances...", flush=True)
    sc_dis = pd.read_csv(
        sc_dis_file, sep='\t', header=0, converters={
            0: ast.literal_eval, 1: int, 2: float, 3: float,
            4: ast.literal_eval},
        names=('idx', 'same_molecule', 'dis_mean', 'dis_med', 'dis')
    ).set_index('idx')
    sc_dis['dis'] = sc_dis.dis.apply(np.array)

    if not scale:
        return sc_dis
    return scale_sc_distances(
        sc_dis, dataset_dir=os.path.dirname(sc_dis_file), copy=False,
        verbose=verbose)


def scale_sc_distances(sc_dis, dataset_dir, copy=True, verbose=True):
    scale_factor = float(pd.read_csv(
        os.path.join(dataset_dir, 'dataset_info.txt'), sep='\t', header=None,
        index_col=0).squeeze("columns")["nghbr_dis_mean.sc_mean"])

    if verbose:
        print(f"Scaling single-cell distances by {scale_factor:g}", flush=True)

    if copy:
        sc_dis = sc_dis.copy()
    for col in ['dis_mean', 'dis_med', 'dis']:
        sc_dis[col] /= scale_factor

    return sc_dis


def get_sc_disterror_all(struct_infer_files, sc_dis=None, hmlg_order_known=None,
                         sc_dis_are_scaled=False, redo=False, verbose=True):
    if isinstance(struct_infer_files, str):
        struct_infer_files = [struct_infer_files]
    dataset_dir = get_dataset_dir(struct_infer_files)

    if sc_dis is None:
        sc_dis = os.path.join(dataset_dir, "dis_true.per_locus.csv")
    if isinstance(sc_dis, str):
        sc_dis = load_sc_distances(sc_dis, scale=True, verbose=verbose).drop(
            'same_molecule', axis=1)
    elif not sc_dis_are_scaled:
        sc_dis = scale_sc_distances(
            sc_dis, dataset_dir=dataset_dir, copy=True, verbose=verbose)

    for struct_infer_file in struct_infer_files:
        compare_infer_vs_true(
            struct_infer_file, sc_dis=sc_dis, hmlg_order_known=hmlg_order_known,
            redo=redo, verbose=verbose)


# Run

In [ ]:
get_sc_disterror_all(
    struct_infer_file, sc_dis=sc_dis, hmlg_order_known=None,
    sc_dis_are_scaled=True, redo=True, verbose=True)

# Temp

In [ ]:
import pandas as pd
import numpy as np
import ast

sc_dis_file = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/cluster.LINK/astro/compare_to_1cell/matrix2d/astro.distances.per_locus.csv'

sc_dis = pd.read_csv(
    sc_dis_file, sep='\t', header=0, converters={
        0: ast.literal_eval, 1: str, 2: float, 3: float, 4: ast.literal_eval},
    names=('idx', 'same_molecule', 'dis_mean', 'dis_med', 'dis'))

sc_dis

sc_dis['same_molecule'] = sc_dis['same_molecule'].astype(int)

sc_dis

sc_dis.to_csv(sc_dis_file, index=False, header=False, sep='\t')

# sed 's/True/1/;s/False/0/' astro.distances.per_locus.csv > astro.distances.per_locus.csv.new